<a href="https://colab.research.google.com/github/Sela80/Data-Science-/blob/main/Taux_de_d%C3%A9sabonnement_des_clients_des_op%C3%A9rateurs_t%C3%A9l%C3%A9coms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# À propos du jeu de données
Contexte
« Anticipez les comportements pour fidéliser vos clients. Analysez toutes les données clients pertinentes et élaborez des programmes de fidélisation ciblés. » [Exemples de jeux de données IBM]

Contenu
Chaque ligne représente un client, chaque colonne contient les attributs du client décrits dans la colonne Métadonnées.

L'ensemble de données comprend des informations sur :

Clients ayant résilié leur contrat au cours du dernier mois – cette colonne s'intitule « Désabonnement ».
Services auxquels chaque client a souscrit : téléphone, lignes multiples, internet, sécurité en ligne, sauvegarde en ligne, protection des appareils, assistance technique, et streaming TV et films
Informations du compte client : durée d’abonnement, contrat, mode de paiement, facturation électronique, frais mensuels et montant total.
Informations démographiques sur les clients : sexe, tranche d’âge et présence d’un conjoint ou de personnes à charge.
Inspiration
Pour explorer ce type de modèles et en apprendre davantage sur le sujet.

Nouvelle version d'IBM :
https://community.ibm.com/community/user/businessanalytics/blogs/steven-macko/2019/07/11/telco-customer-churn-1113

In [ ]:
#Importation des bibliothèques nécessaires
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
!pip install kagglehub
!pip install catboost
!pip install kaggle

# **1. Exploitation de la Dataset**

In [ ]:
import kagglehub

# Télécharge le dataset et renvoie le chemin local
path = kagglehub.dataset_download("blastchar/telco-customer-churn")

print("Chemin du dataset :", path)

In [ ]:
import os

# Construire le chemin complet du fichier CSV en rejoignant le chemin du dossier avec le nom du fichier
csv_file_name = 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
full_csv_path = os.path.join(path, csv_file_name)

# Chargement
df = pd.read_csv(full_csv_path)
display(df.head())

In [ ]:


# Créer CLV tout de suite après le chargement, une seule fois
df['CLV'] = df['MonthlyCharges'] * df['tenure']

df[['MonthlyCharges', 'tenure', 'CLV']].head()
df['CLV'].describe()

df.head()


In [ ]:
df.columns

In [ ]:

df.info()

In [ ]:
df.isna().sum()          # à ce stade TotalCharges est encore 'object' → résultat trompeur, voir plus bas


# ==========================================
# AUDIT QUALITÉ DES DONNÉES
# ==========================================


In [ ]:

# 1. DOUBLONS
print("Doublons complets :", df.duplicated().sum())
print("Doublons sur customerID :", df['customerID'].duplicated().sum())



In [ ]:
# 2. IRRÉGULARITÉS LEXICALES — valeurs incohérentes dans les catégorielles
cat_cols_check = df.select_dtypes(include='object').columns.tolist()
for col in cat_cols_check:
    print(f"\n{col} :")
    print(df[col].unique())

In [ ]:
# 3. FORMATAGE — types de données incohérents
print(df.dtypes)
# TotalCharges doit être object à ce stade (texte) à cause des 11 lignes avec espace " "
# C'est LE problème de formatage déjà identifié — confirme-le explicitement :
espaces_vides = df[df['TotalCharges'].str.strip() == '']
print("Lignes avec TotalCharges vide/espace :", len(espaces_vides))
print(espaces_vides[['customerID', 'tenure', 'TotalCharges']])

In [ ]:
# 3. Nettoyer TotalCharges D'ABORD
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("Valeurs manquantes après conversion :", df['TotalCharges'].isna().sum())
df['TotalCharges'] = df['TotalCharges'].fillna(0)


In [ ]:

# 4. MAINTENANT tu peux détecter les outliers
def detect_outliers_iqr(data, col):
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = data[(data[col] < lower) | (data[col] > upper)]
    print(f"{col} : {len(outliers)} outliers détectés (bornes : {lower:.2f} à {upper:.2f})")
    return outliers

for col in ['tenure', 'MonthlyCharges', 'TotalCharges', 'CLV']:
    detect_outliers_iqr(df, col)

In [ ]:
# 5. VISUALISATION DES OUTLIERS — boxplots
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for i, col in enumerate(['tenure', 'MonthlyCharges', 'TotalCharges', 'CLV']):
    sns.boxplot(y=df[col], ax=axes[i])
    axes[i].set_title(col)
plt.tight_layout()
plt.show()

In [ ]:

df.describe().transpose()


#  Préparation des données

In [ ]:
df1 = df.copy()

In [ ]:
# 2. Exclure fuite + colonnes sans valeur prédictive
cols_to_exclude = ['customerID', 'Churn', 'CLV', 'MonthlyCharges', 'tenure', 'TotalCharges']
x = df1.drop(columns=cols_to_exclude)
y = df1['CLV']


In [ ]:
# Matrice de corrélation
numerical_cols = ['MonthlyCharges', 'tenure', 'TotalCharges', 'CLV']
correlation_matrix = df[numerical_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Matrice de corrélation — MonthlyCharges, tenure, TotalCharges, CLV')
plt.show()

In [ ]:
# 4. Colonnes catégorielles à partir de x
cat_cols = x.select_dtypes(include="object").columns.tolist()
cat_cols = cat_cols + ['SeniorCitizen']
print("Colonnes catégorielles :", cat_cols)
print("Shape final de x :", x.shape)


In [ ]:
# 5. Split
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)


In [ ]:
# 6. Entraînement
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

model_CatBoost_R = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    random_seed=42,
    verbose=100
)
model_CatBoost_R.fit(
    x_train, y_train, cat_features=cat_cols,
    eval_set=(x_test, y_test), early_stopping_rounds=50
)


In [ ]:
from catboost import Pool, cv

# Créer le Pool avec toutes les données (x, y complets, pas juste train)
pool = Pool(x, y, cat_features=cat_cols)

params = {
    'iterations': 500,
    'learning_rate': 0.05,
    'depth': 6,
    'loss_function': 'RMSE',
    'random_seed': 42,
    'verbose': False
}

cv_results = cv(
    pool,
    params,
    fold_count=5,
    partition_random_seed=42,
    shuffle=True,
    verbose=False
)

print(cv_results.tail())
print("\nRMSE moyen (test) :", cv_results['test-RMSE-mean'].iloc[-1])
print("RMSE écart-type (test) :", cv_results['test-RMSE-std'].iloc[-1])

In [ ]:
# 7. Évaluation
y_pred = model_CatBoost_R.predict(x_test)
print("R² :", r2_score(y_test, y_pred))
print("RMSE :", np.sqrt(mean_squared_error(y_test, y_pred)))
print("MAE :", mean_absolute_error(y_test, y_pred))


In [ ]:
importances = model_CatBoost_R.get_feature_importance(prettified=True)
print(importances)

# Visualisation
plt.figure(figsize=(10, 6))
sns.barplot(data=importances, x='Importances', y='Feature Id', hue='Feature Id', palette='viridis', legend=False)
plt.title('Importance des variables — Prédiction du CLV')
plt.xlabel('Importance (%)')
plt.tight_layout()
plt.show()

In [ ]:
# 8. Sauvegarde
model_CatBoost_R.save_model('model_CatBoost_R.cbm')
print('Modèle sauvegardé')


In [ ]:
from google.colab import files
files.download('model_CatBoost_R.cbm')